# DSN Bootcamp Qualification Hackathon 2026 — ML Track
### Predicting product-store `total_sales` for DSN Mart





In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
import lightgbm as lgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 1. Load the data


In [2]:
train = pd.read_csv("/mnt/user-data/uploads/train.csv")
test = pd.read_csv("/mnt/user-data/uploads/test.csv")

TARGET = "total_sales"
CAT_COLS = ["fat_content", "product_category", "store_code",
            "store_size", "store_location_tier", "store_format"]
NUM_COLS = ["product_weight_kg", "shelf_visibility", "product_price", "store_age_years"]

print("train:", train.shape, " test:", test.shape)
train.head()

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/uploads/train.csv'

## 2. Exploratory data analysis




In [ ]:
print("Missing values (train):")
print(train.isna().sum())
print()
print("Missing values (test):")
print(test.isna().sum())

In [ ]:
# product_category is spelled inconsistently across rows 
train["product_category"].value_counts().head(10)

In [ ]:
# Check if is store_size is inconsistent for the *same* store_code
full_check = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
store_size_by_store = full_check.groupby("store_code")["store_size"].agg(lambda s: s.dropna().unique())
store_size_by_store

In [ ]:
# Check which stores have store_size missing for 100% of their rows
missing_stores = full_check[full_check.store_size.isna()].store_code.value_counts()
missing_stores

In [ ]:
# Describe the target distribution
train[TARGET].describe()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(train[TARGET], bins=40)
plt.title("Distribution of total_sales (train)")
plt.xlabel("total_sales")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## 3. Cleaning

- **`product_category`**: canonicalize casing (`"CANNED"` / `"canned"` / `"Canned"` → `"Canned"`).
- **`product_weight_kg`** (missing ~18%): the same `product_code` is sold across several stores and should
  weigh the same everywhere (small variation observed is measurement noise) → impute from the product's
  own median weight, falling back to the category median for the rare product with no known weight at all.
- **`store_size`** (missing ~28%): traced to specific stores that have **zero** known size across *every*
  one of their rows — this isn't recoverable, so it's kept as an explicit `"Missing"` category rather than
  guessed.
- **`shelf_visibility` == 0**: physically implausible (a stocked product has *some* shelf visibility) →
  treated as missing and imputed from the product category's average visibility.


In [ ]:
train["is_train"] = 1
test["is_train"] = 0
test[TARGET] = np.nan
full = pd.concat([train, test], ignore_index=True, sort=False)

# 3a. Canonicalize category casing
full["product_category"] = full["product_category"].str.strip().str.lower().str.title()

# 3b. Impute product_weight_kg using product's own median
prod_weight_map = full.groupby("product_code")["product_weight_kg"].median()
full["product_weight_kg"] = full["product_weight_kg"].fillna(full["product_code"].map(prod_weight_map))
cat_weight_map = full.groupby("product_category")["product_weight_kg"].median()
full["product_weight_kg"] = full["product_weight_kg"].fillna(full["product_category"].map(cat_weight_map))

# 3c. store_size: keep missingness as its own category
full["store_size"] = full["store_size"].fillna("Missing")

# 3d. shelf_visibility == 0 -> treat as missing, impute from category mean
full.loc[full["shelf_visibility"] == 0, "shelf_visibility"] = np.nan
vis_cat_map = full.groupby("product_category")["shelf_visibility"].transform("mean")
full["shelf_visibility"] = full["shelf_visibility"].fillna(vis_cat_map)

print("Remaining missing values:")
print(full[CAT_COLS + NUM_COLS].isna().sum())

## 4. Feature engineering

- `price_per_kg`: normalizes price by product weight.
- `visibility_ratio`: how a product's shelf visibility compares to its category's average
  (>1 = more visible than typical for that category).
- `product_store_reach`: how many distinct stores carry this product (a proxy for how mainstream it is).


In [ ]:
full["price_per_kg"] = full["product_price"] / full["product_weight_kg"]

cat_vis_mean = full.groupby("product_category")["shelf_visibility"].transform("mean")
full["visibility_ratio"] = full["shelf_visibility"] / cat_vis_mean

full["product_store_reach"] = full.groupby("product_code")["store_code"].transform("nunique")

FEATURES = CAT_COLS + NUM_COLS + ["price_per_kg", "visibility_ratio", "product_store_reach"]

train = full[full["is_train"] == 1].copy()
test = full[full["is_train"] == 0].copy()

for c in CAT_COLS:
    train[c] = train[c].astype("category")
    test[c] = test[c].astype(pd.CategoricalDtype(categories=train[c].cat.categories))

X = train[FEATURES]
y = train[TARGET].values
X_test = test[FEATURES]

print("X:", X.shape, " X_test:", X_test.shape)
X.dtypes

## 5. Model comparison (5-fold cross-validation)

Comparing a linear baseline, Random Forest, LightGBM, and CatBoost 


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cat_idx = [X.columns.get_loc(c) for c in CAT_COLS]

def rmse(a, b):
    return mean_squared_error(a, b) ** 0.5

results = {}

In [ ]:
#Baseline: Ridge on one-hot encoded features 
X_ohe_ready = X.copy()
for c in CAT_COLS:
    X_ohe_ready[c] = X_ohe_ready[c].astype(str)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_cat_ohe = ohe.fit_transform(X_ohe_ready[CAT_COLS])
X_num = X_ohe_ready[NUM_COLS + ["price_per_kg", "visibility_ratio", "product_store_reach"]].values
X_ridge_full = hstack([X_cat_ohe, X_num]).tocsr()

scores = []
for tr_idx, va_idx in kf.split(X_ridge_full):
    model = Ridge(alpha=5.0, random_state=RANDOM_STATE)
    model.fit(X_ridge_full[tr_idx], y[tr_idx])
    pred = model.predict(X_ridge_full[va_idx])
    scores.append(rmse(y[va_idx], pred))
results["Ridge (baseline)"] = np.mean(scores)
print("Ridge RMSE:", results["Ridge (baseline)"])

In [ ]:
# Random Forest 
X_rf = X.copy()
for c in CAT_COLS:
    X_rf[c] = X_rf[c].cat.codes

scores = []
for tr_idx, va_idx in kf.split(X_rf):
    model = RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_leaf=5,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_rf.iloc[tr_idx], y[tr_idx])
    pred = model.predict(X_rf.iloc[va_idx])
    scores.append(rmse(y[va_idx], pred))
results["Random Forest"] = np.mean(scores)
print("Random Forest RMSE:", results["Random Forest"])

In [ ]:
# LightGBM: raw target vs log1p target 
lgb_params = dict(
    n_estimators=2000, learning_rate=0.02, num_leaves=15,
    min_child_samples=25, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1
)

for use_log in [False, True]:
    scores = []
    for tr_idx, va_idx in kf.split(X):
        model = lgb.LGBMRegressor(**lgb_params)
        y_tr = np.log1p(y[tr_idx]) if use_log else y[tr_idx]
        model.fit(X.iloc[tr_idx], y_tr, categorical_feature=CAT_COLS,
                  eval_set=[(X.iloc[va_idx], np.log1p(y[va_idx]) if use_log else y[va_idx])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        pred = model.predict(X.iloc[va_idx])
        if use_log:
            pred = np.expm1(pred)
        scores.append(rmse(y[va_idx], pred))
    results[f"LightGBM (log_target={use_log})"] = np.mean(scores)

print({k: v for k, v in results.items() if "LightGBM" in k})

In [ ]:
# CatBoost: raw target vs log1p target
for use_log in [False, True]:
    scores = []
    for tr_idx, va_idx in kf.split(X):
        model = CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                                   l2_leaf_reg=3.0, random_state=RANDOM_STATE,
                                   cat_features=cat_idx, verbose=False,
                                   early_stopping_rounds=100)
        y_tr = np.log1p(y[tr_idx]) if use_log else y[tr_idx]
        y_va = np.log1p(y[va_idx]) if use_log else y[va_idx]
        model.fit(X.iloc[tr_idx], y_tr, eval_set=(X.iloc[va_idx], y_va))
        pred = model.predict(X.iloc[va_idx])
        if use_log:
            pred = np.expm1(pred)
        scores.append(rmse(y[va_idx], pred))
    results[f"CatBoost (log_target={use_log})"] = np.mean(scores)

print("\n=== 5-fold CV RMSE (lower is better) ===")
for k, v in sorted(results.items(), key=lambda kv: kv[1]):
    print(f"{k:35s} {v:10.2f}")

**Takeaway:** CatBoost on the raw target is the clear winner. The `log1p` transform actually hurts
here — makes sense, since we're optimizing/evaluating RMSE on the original scale, not the log scale.

## 6. Does blending CatBoost + LightGBM help?



In [ ]:
oof_cb = np.zeros(len(y))
oof_lgb = np.zeros(len(y))

for tr_idx, va_idx in kf.split(X):
    m_cb = CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                              l2_leaf_reg=3.0, random_state=RANDOM_STATE,
                              cat_features=cat_idx, verbose=False,
                              early_stopping_rounds=100)
    m_cb.fit(X.iloc[tr_idx], y[tr_idx], eval_set=(X.iloc[va_idx], y[va_idx]))
    oof_cb[va_idx] = m_cb.predict(X.iloc[va_idx])

    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X.iloc[tr_idx], y[tr_idx], categorical_feature=CAT_COLS,
              eval_set=[(X.iloc[va_idx], y[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[va_idx] = m_lgb.predict(X.iloc[va_idx])

best_w, best_rmse = None, 1e18
for w in np.arange(0.0, 1.01, 0.1):
    blend = w * oof_cb + (1 - w) * oof_lgb
    r = rmse(y, blend)
    if r < best_rmse:
        best_rmse, best_w = r, w

print(f"Best blend: {best_w:.1f}*CatBoost + {1-best_w:.1f}*LightGBM -> OOF RMSE {best_rmse:.2f}")
print(f"CatBoost alone OOF RMSE: {rmse(y, oof_cb):.2f}")
print(f"LightGBM alone OOF RMSE: {rmse(y, oof_lgb):.2f}")

Blending doesn't beat CatBoost alone, so the final model stays simple: **CatBoost only**.

## 7. Final model




In [ ]:
FINAL_PARAMS = dict(depth=4, learning_rate=0.03, l2_leaf_reg=3.0)

best_iters = []
final_oof = np.zeros(len(y))
for tr_idx, va_idx in kf.split(X):
    m = CatBoostRegressor(iterations=3000, random_state=RANDOM_STATE, cat_features=cat_idx,
                           verbose=False, early_stopping_rounds=150, **FINAL_PARAMS)
    m.fit(X.iloc[tr_idx], y[tr_idx], eval_set=(X.iloc[va_idx], y[va_idx]))
    best_iters.append(m.get_best_iteration())
    final_oof[va_idx] = m.predict(X.iloc[va_idx])

final_cv_rmse = rmse(y, final_oof)
n_estimators_final = int(np.mean(best_iters) * 1.1)  # small cushion since trained on more data (full train set)
print(f"Final model OOF RMSE: {final_cv_rmse:.2f}")
print(f"Mean best_iteration across folds: {np.mean(best_iters):.0f} -> using {n_estimators_final} for full-data fit")

In [ ]:
final_model = CatBoostRegressor(iterations=n_estimators_final, random_state=RANDOM_STATE,
                                 cat_features=cat_idx, verbose=False, **FINAL_PARAMS)
final_model.fit(X, y)

test_pred = final_model.predict(X_test)
test_pred = np.clip(test_pred, 0, None)  # sales cannot be negative

submission = pd.DataFrame({"id": test["id"].values, "total_sales": test_pred})
submission.to_csv("/mnt/user-data/outputs/submission.csv", index=False)
print("Saved submission:", submission.shape)
submission.head()

## 8. Check Feature Importances

In [ ]:
importances = pd.Series(final_model.get_feature_importance(), index=FEATURES).sort_values()

plt.figure(figsize=(7,5))
plt.barh(importances.index, importances.values)
plt.title("CatBoost feature importance")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

importances.sort_values(ascending=False)

`product_price` and `store_format` together account for the large majority of the model's prediction power. Item's price and the type of outlet it's sold through explain most of the variance in sales, while item with these attributes (weight, category, fat content, visibility) play a much smaller supporting role.


## 9. Summary

- Cleaned inconsistent category labels, imputed missing weights/visibility from product/category norms,and kept unknown `store_size` as its own category rather than guessing.
- Compared 6 model configurations via 5-fold CV; **CatBoost on the target** was the clear, stable winner.
- Final 5-fold CV RMSE ≈ **1075**.
